# Enterprise Private RAG — лабораторная тетрадь (исправленная версия)

Основано на паттернах из курса LangChain (BaseChatBot / create_llm / LCEL),
адаптировано под задачу: **локальный RAG на Ollama + Qdrant + HuggingFace embeddings**.

Обновления в этой версии:
- используется `langchain_ollama.ChatOllama` (а не устаревший `langchain_community`);
- `reasoning=False`, `num_ctx=2048`, `num_predict=1024` — отключение "thinking"-режима у qwen3.5 и ускорение на CPU;
- поддержка отдельной **тестовой коллекции** Qdrant (`QDRANT_TEST_COLLECTION`) и тестового корпуса (`TEST_DOCS_DIR`), чтобы не путать с рабочей базой;
- логирование входа/выхода LLM.

Конвертация в `.py`-скрипт:
```bash
jupytext --to py:percent notebook/rag_lab.ipynb
```
Подробности — в `docs/NOTEBOOK_CONVERSION.md`.


## Как запустить в VS Code

1. Откройте папку проекта `enterprise_rag_langchain/` в VS Code (`File -> Open Folder`).
2. Убедитесь, что venv уже создан и зависимости установлены в терминале:
   ```bash
   python3 -m venv venv
   source venv/bin/activate      # Windows: venv\Scripts\activate
   pip install -r requirements.txt
   pip install ipykernel
   ```
3. Установите расширения **Python** и **Jupyter** от Microsoft (Extensions: Ctrl+Shift+X).
4. Выберите интерпретатор: `Ctrl+Shift+P` -> **Python: Select Interpreter** -> укажите путь
   `./venv/bin/python` (Linux/Mac) или `.\venv\Scripts\python.exe` (Windows).
5. Откройте `notebook/rag_lab.ipynb`. В правом верхнем углу нажмите **Select Kernel**
   -> **Python Environments** -> выберите тот же venv (обычно подхватывается автоматически
   после шага 4, если venv лежит внутри открытой папки проекта).
6. Если venv не появляется в списке ядер, зарегистрируйте его явно:
   ```bash
   python -m ipykernel install --user --name=enterprise_rag_venv --display-name "Python (enterprise_rag venv)"
   ```
   и выберите "Python (enterprise_rag venv)" в селекторе ядра.
7. Запускайте ячейки по порядку (`Shift+Enter`). Перед этим убедитесь, что Ollama и Qdrant
   (`docker compose up -d`) уже запущены.

**venv-kernel — рекомендуемый выбор**: так notebook использует те же
пакеты (`langchain-ollama`, `qdrant-client` и т.д.), что установлены в venv для src/-кода,
и не будет конфликтов версий с системным Python.


In [2]:
# Проверка, что ядро - это venv (путь должен указывать на venv/bin/python или venv\Scripts\python.exe)
import sys
print(sys.executable)


/home/rtg/github/gritsulyak/llm-learn-tmp-2026/lessons/04_l11_langchain_RAG/venv/bin/python


In [3]:
# Установка зависимостей (если запускаете вне venv с requirements.txt)
# !pip install -q -r ../requirements.txt


In [4]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

from src.config import (
    OLLAMA_BASE_URL, OLLAMA_LLM_MODEL, EMBEDDING_MODEL,
    QDRANT_URL, QDRANT_COLLECTION, QDRANT_TEST_COLLECTION,
    CHUNK_SIZE, CHUNK_OVERLAP, TOP_K, DOCS_DIR, TEST_DOCS_DIR,
)

print("Ollama model:", OLLAMA_LLM_MODEL)
print("Embeddings:", EMBEDDING_MODEL)
print("Qdrant:", QDRANT_URL, "| рабочая коллекция:", QDRANT_COLLECTION, "| тестовая:", QDRANT_TEST_COLLECTION)
print("Docs dir (рабочий):", DOCS_DIR)
print("Docs dir (тестовый):", TEST_DOCS_DIR)


Ollama model: llama3:latest
Embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Qdrant: http://localhost:6333 | рабочая коллекция: enterprise_docs | тестовая: enterprise_docs_test
Docs dir (рабочий): /home/rtg/github/gritsulyak/llm-learn-tmp-2026/lessons/04_l11_langchain_RAG/docs
Docs dir (тестовый): /home/rtg/github/gritsulyak/llm-learn-tmp-2026/lessons/04_l11_langchain_RAG/docs_corpus_sample


## 1. Проверка локальной LLM (Ollama)

In [5]:
from src.rag_chain import create_llm

llm = create_llm(OLLAMA_LLM_MODEL)
resp = llm.invoke("Ответь одним словом: ты работаешь локально без интернета?")
print(resp.content)


2026-08-02 13:39:52,181 | INFO | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Да.


## 2. Загрузка и чанкование документов (тестовый корпус)\n\nИспользуем `TEST_DOCS_DIR` (docs_corpus_sample/) и отдельную `QDRANT_TEST_COLLECTION`, чтобы не трогать вашу рабочую базу знаний.

In [6]:
from src.ingest import run_ingest

run_ingest(TEST_DOCS_DIR, QDRANT_TEST_COLLECTION)


Гружу из: /home/rtg/github/gritsulyak/llm-learn-tmp-2026/lessons/04_l11_langchain_RAG/docs_corpus_sample


100%|██████████| 1/1 [00:00<00:00,  9.48it/s]

Загружено документов: 7
Получено чанков: 18 (chunk_size=1800, overlap=120)



2026-08-02 13:39:54,396 | INFO | Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
2026-08-02 13:39:57,949 | INFO | HTTP Request: GET http://localhost:6333/collections/enterprise_docs_test/exists "HTTP/1.1 200 OK"
2026-08-02 13:39:57,955 | INFO | HTTP Request: GET http://localhost:6333/collections/enterprise_docs_test "HTTP/1.1 200 OK"
2026-08-02 13:39:58,276 | INFO | HTTP Request: PUT http://localhost:6333/collections/enterprise_docs_test/points?wait=true "HTTP/1.1 200 OK"


✅ 18 чанков сохранено в Qdrant (коллекция 'enterprise_docs_test').


## 3. Сборка RAG-цепочки (LCEL) на тестовой коллекции

In [7]:
from src.rag_chain import RagBot

bot = RagBot(model_name=OLLAMA_LLM_MODEL, top_k=TOP_K, collection_name=QDRANT_TEST_COLLECTION)


2026-08-02 13:39:58,327 | INFO | Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
2026-08-02 13:40:01,706 | INFO | HTTP Request: GET http://localhost:6333/collections/enterprise_docs_test "HTTP/1.1 200 OK"


## 4. Тестирование: три типа вопросов (см. docs/TASK_RU.md п.4)

In [8]:
result = bot.ask("Какой регламент действий при инциденте информационной безопасности?")
print(result["answer"])
print(result["sources"], result["latency_sec"], "сек")


2026-08-02 13:40:01,758 | INFO | HTTP Request: POST http://localhost:6333/collections/enterprise_docs_test/points/query "HTTP/1.1 200 OK"
2026-08-02 13:40:01,761 | INFO | LLM INPUT >>> System: Ты корпоративный ассистент. Отвечай на вопрос пользователя ТОЛЬКО опираясь на следующий контекст, полученный из внутренних документов компании.
Если ответа в контексте нет — прямо скажи, что не знаешь, и не придумывай факты.
Отвечай сразу, без внутренних рассуждений. /no_think

После ответа ОБЯЗАТЕЛЬНО добавь строку в формате:
ИСПОЛЬЗОВАНО: <название файла>, стр. <номер> (перечисли только те источники из контекста,
на которые ты реально опирался при ответе; если использовал несколько - перечисли через запятую)

Контекст:
[Источник: security_policy_sample.txt]
Регламент реагирования на инциденты информационной безопасности (пример)

1. При обнаружении инцидента сотрудник обязан немедленно уведомить службу ИБ по адресу security@company.local.
2. Ответственный за реагирование фиксирует время обнаруж

При обнаружении инцидента сотрудник обязан немедленно уведомить службу ИБ по адресу security@company.local. Ответственный за реагирование фиксирует время обнаружения и характер инцидента в системе учёта инцидентов. В течение 1 часа формируется команда реагирования, состоящая из представителя ИБ, ИТ-администратора и владельца затронутой системы.
['security_policy_sample.txt'] 32.63 сек


In [9]:
result = bot.ask("Дай рецепт классического яблочного пирога.")
print(result["answer"])


2026-08-02 13:40:34,394 | INFO | HTTP Request: POST http://localhost:6333/collections/enterprise_docs_test/points/query "HTTP/1.1 200 OK"


2026-08-02 13:40:34,396 | INFO | LLM INPUT >>> System: Ты корпоративный ассистент. Отвечай на вопрос пользователя ТОЛЬКО опираясь на следующий контекст, полученный из внутренних документов компании.
Если ответа в контексте нет — прямо скажи, что не знаешь, и не придумывай факты.
Отвечай сразу, без внутренних рассуждений. /no_think

После ответа ОБЯЗАТЕЛЬНО добавь строку в формате:
ИСПОЛЬЗОВАНО: <название файла>, стр. <номер> (перечисли только те источники из контекста,
на которые ты реально опирался при ответе; если использовал несколько - перечисли через запятую)

Контекст:
[Источник: Салтыков-Щедрин Михаил. Соседи - royallib.ru.txt]
– Надо бы, однако, нам это дело рассудить:

– И я говорю: надо бы.

Зевнет и Иван Бедный с своей стороны, перекрестит рот, пойдет спать и во сне завтрашние пустые щи видит. А на другой день проснется – смотрит, Иван Богатый сюрприз ему приготовил: убоины ради праздника во щи прислал.

В следующий предпраздничный канун опять сойдутся соседи и опять за стар

Я не знаю, потому что в контексте нет информации о рецепте классического яблочного пирога. Я могу предложить вам несколько вариантов, но они не основаны на конкретном источнике из контекста.

Однако, я могу дать вам ссылку на Lib.ru_Классика_ Андерсен Ганс Христиан. Новый наряд короля.pdf, который является частью контекста, но в нем нет информации о рецепте классического яблочного пирога.

Если вы хотите получить рецепт классического яблочного пирога, я могу предложить вам несколько вариантов из интернета или других источников.


In [10]:
result = bot.ask("Сколько дней отпуска положено сотруднику по стандартным нормам трудового права?")
print(result["answer"])
print(result["sources"])


2026-08-02 13:41:27,909 | INFO | HTTP Request: POST http://localhost:6333/collections/enterprise_docs_test/points/query "HTTP/1.1 200 OK"
2026-08-02 13:41:27,910 | INFO | LLM INPUT >>> System: Ты корпоративный ассистент. Отвечай на вопрос пользователя ТОЛЬКО опираясь на следующий контекст, полученный из внутренних документов компании.
Если ответа в контексте нет — прямо скажи, что не знаешь, и не придумывай факты.
Отвечай сразу, без внутренних рассуждений. /no_think

После ответа ОБЯЗАТЕЛЬНО добавь строку в формате:
ИСПОЛЬЗОВАНО: <название файла>, стр. <номер> (перечисли только те источники из контекста,
на которые ты реально опирался при ответе; если использовал несколько - перечисли через запятую)

Контекст:
[Источник: security_policy_sample.txt]
Регламент реагирования на инциденты информационной безопасности (пример)

1. При обнаружении инцидента сотрудник обязан немедленно уведомить службу ИБ по адресу security@company.local.
2. Ответственный за реагирование фиксирует время обнаруж

Стандартный отпуск сотрудника компании составляет 32 календарных дня в год (больше нормы ТК РФ в 28 дней) — уточняйте у HR.
['security_policy_sample.txt', 'стр. 4']


## 5. Эксперимент с Top-K (влияние на качество и latency)

In [11]:
import pandas as pd

question = "Какой регламент действий при инциденте информационной безопасности?"
rows = []
for k in (1, 2, 4, 6, 8):
    bot.set_top_k(k)
    r = bot.ask(question)
    rows.append({"top_k": k, "latency_sec": r["latency_sec"], "num_chunks": r["num_chunks"], "answer_len": len(r["answer"])})

df = pd.DataFrame(rows)
df


2026-08-02 13:41:42,103 | INFO | Load pretrained SentenceTransformer: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


2026-08-02 13:41:45,641 | INFO | HTTP Request: GET http://localhost:6333/collections/enterprise_docs_test "HTTP/1.1 200 OK"
2026-08-02 13:41:45,683 | INFO | HTTP Request: POST http://localhost:6333/collections/enterprise_docs_test/points/query "HTTP/1.1 200 OK"
2026-08-02 13:41:45,685 | INFO | LLM INPUT >>> System: Ты корпоративный ассистент. Отвечай на вопрос пользователя ТОЛЬКО опираясь на следующий контекст, полученный из внутренних документов компании.
Если ответа в контексте нет — прямо скажи, что не знаешь, и не придумывай факты.
Отвечай сразу, без внутренних рассуждений. /no_think

После ответа ОБЯЗАТЕЛЬНО добавь строку в формате:
ИСПОЛЬЗОВАНО: <название файла>, стр. <номер> (перечисли только те источники из контекста,
на которые ты реально опирался при ответе; если использовал несколько - перечисли через запятую)

Контекст:
[Источник: security_policy_sample.txt]
Регламент реагирования на инциденты информационной безопасности (пример)

1. При обнаружении инцидента сотрудник обяз

,top_k,latency_sec,num_chunks,answer_len
0,1,16.09,1,235
1,2,24.87,2,346
2,4,26.21,4,346
3,6,25.04,6,346
4,8,29.71,8,346


## 6. Работа с вашей рабочей базой (не тестовой)

Для (`docs/`), проиндексированы документы по тренингам, несколько книг, и статей на техническую тему.


## 7. Выводы


Выбрал модель embedding (HF MiniLM-L12) - как компромис между скоростью и качеством для локальной машины без GPU. Llama3 была выбрана, так как qwen долго и много думал.

Из трудностей: Проблемы с кодировками на ingest стадии. Полностью загруженый CPU блокировал работу по сути на фоне. Thinking не получилось отключить в Qwen, очень долго производил ответ с нерелевантным текстом. Замена на llama - ускорило.Галлюцинации - чанки отдаваемые similiarity search не всегда реально соответствуют смыслу запроса; в тестовом сете например выдаются фрагменты сказок, при запросе рецепта яблочного пирога, но это затем отсекается.

Увеличение k в top-k может улучшить качество и полноту выдачи, но требует больше времени для llm чтобы обработать запрос.

В целом система работает приемлемо для личных целей, но скорость и использование ресурсов при поиске гораздо слабее, чем например использование десктопных поисковиков по точной подстроке (типа recoll для linux).

Вместо наивного подхода лучше конечно использовать более продвинутые подходы, с построением графа.

Screencast: https://youtu.be/RvSposzRNf0  